In [ ]:
import os
import joblib
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    roc_curve
)

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully.")


Libraries imported successfully.


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Google Colab is not available in this environment; skipping Drive mount.")


Mounted at /content/drive


In [ ]:

print("Files in current directory:")
for file in os.listdir():
    print(file)

Files in current directory:
.config
drive
sample_data


In [ ]:
DATA_ROOT = os.environ.get("NSL_KDD_DATA_DIR", "/content/drive/MyDrive/ML-DP-NID")
TRAIN_FILE = os.path.join(DATA_ROOT, "KDDTrain+.txt")
TEST_FILE = os.path.join(DATA_ROOT, "KDDTest+.txt")

assert os.path.exists(TRAIN_FILE), f"{TRAIN_FILE} not found."
assert os.path.exists(TEST_FILE), f"{TEST_FILE} not found."

print("Train file found:", TRAIN_FILE)
print("Test file found:", TEST_FILE)


Train file found: /content/drive/MyDrive/ML-DP-NID/KDDTrain+.txt
Test file found: /content/drive/MyDrive/ML-DP-NID/KDDTest+.txt


In [ ]:
feature_columns = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins",
    "logged_in", "num_compromised", "root_shell", "su_attempted", "num_root",
    "num_file_creations", "num_shells", "num_access_files", "num_outbound_cmds",
    "is_host_login", "is_guest_login", "count", "srv_count", "serror_rate",
    "srv_serror_rate", "rerror_rate", "srv_rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate"
]

columns_43 = feature_columns + ["label", "difficulty"]

print("Total columns:", len(columns_43))

Total columns: 43


In [ ]:
train_df = pd.read_csv(
    TRAIN_FILE,
    sep=",",
    names=columns_43,
    header=None
)

test_df = pd.read_csv(
    TEST_FILE,
    sep=",",
    names=columns_43,
    header=None
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())
display(test_df.head())

Train shape: (125973, 43)
Test shape: (22544, 43)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,private,REJ,0,0,0,0,0,0,...,0.04,0.06,0.00,0.00,0.0,0.0,1.00,1.00,neptune,21
1,0,tcp,private,REJ,0,0,0,0,0,0,...,0.00,0.06,0.00,0.00,0.0,0.0,1.00,1.00,neptune,21
2,2,tcp,ftp_data,SF,12983,0,0,0,0,0,...,0.61,0.04,0.61,0.02,0.0,0.0,0.00,0.00,normal,21
3,0,icmp,eco_i,SF,20,0,0,0,0,0,...,1.00,0.00,1.00,0.28,0.0,0.0,0.00,0.00,saint,15
4,1,tcp,telnet,RSTO,0,15,0,0,0,0,...,0.31,0.17,0.03,0.02,0.0,0.0,0.83,0.71,mscan,11


In [ ]:
print("Train missing values:", train_df.isna().sum().sum())
print("Test missing values:", test_df.isna().sum().sum())

print("\nTrain duplicate rows:", train_df.duplicated().sum())
print("Test duplicate rows:", test_df.duplicated().sum())

print("\nTrain labels:")
print(train_df["label"].value_counts().head(20))

print("\nTest labels:")
print(test_df["label"].value_counts().head(20))

print("\nDifficulty values:")
print(train_df["difficulty"].value_counts().head())

Train missing values: 0
Test missing values: 0

Train duplicate rows: 0
Test duplicate rows: 0

Train labels:
label
normal             67343
neptune            41214
satan               3633
ipsweep             3599
portsweep           2931
smurf               2646
nmap                1493
back                 956
teardrop             892
warezclient          890
pod                  201
guess_passwd          53
buffer_overflow       30
warezmaster           20
land                  18
imap                  11
rootkit               10
loadmodule             9
ftp_write              8
multihop               7
Name: count, dtype: int64

Test labels:
label
normal             9711
neptune            4657
guess_passwd       1231
mscan               996
warezmaster         944
apache2             737
satan               735
processtable        685
smurf               665
back                359
snmpguess           331
saint               319
mailbomb            293
snmpgetattack       178
po

In [ ]:
def clean_text_value(value):
    value = str(value).strip().lower()
    if value.endswith("."):
        value = value[:-1]
    return value

train_df["label"] = train_df["label"].apply(clean_text_value)
test_df["label"] = test_df["label"].apply(clean_text_value)

for col in ["protocol_type", "service", "flag"]:
    train_df[col] = train_df[col].apply(clean_text_value)
    test_df[col] = test_df[col].apply(clean_text_value)

print("Cleaned train labels:")
print(train_df["label"].value_counts().head(20))

print("\nCleaned test labels:")
print(test_df["label"].value_counts().head(20))


Cleaned train labels:
label
normal             67343
neptune            41214
satan               3633
ipsweep             3599
portsweep           2931
smurf               2646
nmap                1493
back                 956
teardrop             892
warezclient          890
pod                  201
guess_passwd          53
buffer_overflow       30
warezmaster           20
land                  18
imap                  11
rootkit               10
loadmodule             9
ftp_write              8
multihop               7
Name: count, dtype: int64

Cleaned test labels:
label
normal             9711
neptune            4657
guess_passwd       1231
mscan               996
warezmaster         944
apache2             737
satan               735
processtable        685
smurf               665
back                359
snmpguess           331
saint               319
mailbomb            293
snmpgetattack       178
portsweep           157
ipsweep             141
httptunnel          133
nmap      

In [ ]:
def to_binary_label(label):
    return 0 if label == "normal" else 1

train_df["binary_label"] = train_df["label"].apply(to_binary_label)
test_df["binary_label"] = test_df["label"].apply(to_binary_label)

print("Train binary labels:")
print(train_df["binary_label"].value_counts())

print("\nTest binary labels:")
print(test_df["binary_label"].value_counts())

Train binary labels:
binary_label
0    67343
1    58630
Name: count, dtype: int64

Test binary labels:
binary_label
1    12833
0     9711
Name: count, dtype: int64


In [ ]:
categorical_cols = ["protocol_type", "service", "flag"]
numeric_cols = [col for col in feature_columns if col not in categorical_cols]

print("Categorical columns:", categorical_cols)
print("Number of numeric columns:", len(numeric_cols))

Categorical columns: ['protocol_type', 'service', 'flag']
Number of numeric columns: 38


In [ ]:
for col in numeric_cols:
    train_df[col] = pd.to_numeric(train_df[col], errors="raise")
    test_df[col] = pd.to_numeric(test_df[col], errors="raise")

print("Train numeric NaNs:", train_df[numeric_cols].isna().sum().sum())
print("Test numeric NaNs:", test_df[numeric_cols].isna().sum().sum())


Train numeric NaNs: 0
Test numeric NaNs: 0


In [ ]:
assert train_df[numeric_cols].isna().sum().sum() == 0, "Train numeric columns contain missing values."
assert test_df[numeric_cols].isna().sum().sum() == 0, "Test numeric columns contain missing values."

print("Numeric columns verified: no missing values.")


Numeric columns verified: no missing values.


In [ ]:
train_part, val_part = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["binary_label"]
)

print("Train part:", train_part.shape)
print("Validation part:", val_part.shape)

print("Train binary label distribution:")
print(train_part["binary_label"].value_counts())

print("\nValidation binary label distribution:")
print(val_part["binary_label"].value_counts())

X_train_raw = train_part.drop(columns=["label", "difficulty", "binary_label"])
y_train = train_part["binary_label"]

X_val_raw = val_part.drop(columns=["label", "difficulty", "binary_label"])
y_val = val_part["binary_label"]

X_test_raw = test_df.drop(columns=["label", "difficulty", "binary_label"])
y_test = test_df["binary_label"]

print("X_train_raw:", X_train_raw.shape)
print("X_val_raw:", X_val_raw.shape)
print("X_test_raw:", X_test_raw.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)


Train part: (100778, 44)
Validation part: (25195, 44)
Train binary label distribution:
binary_label
0    53874
1    46904
Name: count, dtype: int64

Validation binary label distribution:
binary_label
0    13469
1    11726
Name: count, dtype: int64
X_train_raw: (100778, 41)
X_val_raw: (25195, 41)
X_test_raw: (22544, 41)
y_train: (100778,)
y_val: (25195,)
y_test: (22544,)


In [ ]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def get_model_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        scores = np.asarray(scores, dtype=float)
        denom = scores.max() - scores.min()
        if denom == 0:
            return np.zeros_like(scores)
        return (scores - scores.min()) / denom
    return None


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", MinMaxScaler(), numeric_cols),
        ("cat", make_one_hot_encoder(), categorical_cols),
    ],
    remainder="drop"
)

X_train_processed = preprocessor.fit_transform(X_train_raw)
X_val_processed = preprocessor.transform(X_val_raw)
X_test_processed = preprocessor.transform(X_test_raw)

print("Processed train shape:", X_train_processed.shape)
print("Processed val shape:", X_val_processed.shape)
print("Processed test shape:", X_test_processed.shape)


Processed train shape: (100778, 121)
Processed val shape: (25195, 121)
Processed test shape: (22544, 121)


In [ ]:
def evaluate_model(
    model_name,
    split_name,
    y_true,
    y_pred,
    y_score=None,
    threshold=0.5,
    experiment="Baseline"
):
    """
    Evaluate binary IDS predictions.

    Positive class = Attack (1)
    Negative class = Normal (0)
    """
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    roc_auc = None
    pr_auc = None
    if y_score is not None:
        y_score = np.asarray(y_score, dtype=float)
        if len(np.unique(y_true)) > 1:
            roc_auc = roc_auc_score(y_true, y_score)
            pr_auc = average_precision_score(y_true, y_score)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    false_negative_rate = fn / (fn + tp) if (fn + tp) else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) else 0.0

    print("=" * 70)
    print(f"{experiment} | {model_name} | {split_name}")
    print("=" * 70)
    print("Threshold:", threshold)
    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1-score:", f1)
    print("ROC-AUC:", roc_auc)
    print("PR-AUC:", pr_auc)
    print("False Negative Rate:", false_negative_rate)
    print("False Positive Rate:", false_positive_rate)

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["Normal", "Attack"],
        zero_division=0
    ))

    return {
        "Experiment": experiment,
        "Model": model_name,
        "Split": split_name,
        "Threshold": threshold,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC_AUC": roc_auc,
        "PR_AUC": pr_auc,
        "False Negative Rate": false_negative_rate,
        "False Positive Rate": false_positive_rate,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
    }


In [ ]:
# ============================================================
# 16. Train and Evaluate Random Forest
# Default decision threshold = 0.5
# ============================================================

results = []
validation_results = []

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_model.fit(X_train_processed, y_train)

rf_val_score = get_model_scores(rf_model, X_val_processed)
rf_test_score = get_model_scores(rf_model, X_test_processed)

rf_val_pred = (rf_val_score >= 0.5).astype(int)
rf_test_pred = (rf_test_score >= 0.5).astype(int)

validation_results.append(
    evaluate_model(
        model_name="Random Forest",
        split_name="Validation",
        y_true=y_val,
        y_pred=rf_val_pred,
        y_score=rf_val_score,
        threshold=0.5,
        experiment="Baseline"
    )
)

results.append(
    evaluate_model(
        model_name="Random Forest",
        split_name="Test",
        y_true=y_test,
        y_pred=rf_test_pred,
        y_score=rf_test_score,
        threshold=0.5,
        experiment="Baseline"
    )
)


Baseline | Random Forest | Validation
Threshold: 0.5
Accuracy: 0.999047430045644
Precision: 0.9994877923851802
Recall: 0.9984649496844619
F1-score: 0.9989761092150171
ROC-AUC: 0.9999936050650519
PR-AUC: 0.9999921927603493
False Negative Rate: 0.0015350503155381204
False Positive Rate: 0.000445467369515183

Confusion Matrix:
[[13463     6]
 [   18 11708]]

Classification Report:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     13469
      Attack       1.00      1.00      1.00     11726

    accuracy                           1.00     25195
   macro avg       1.00      1.00      1.00     25195
weighted avg       1.00      1.00      1.00     25195

Baseline | Random Forest | Test
Threshold: 0.5
Accuracy: 0.7804737402413059
Precision: 0.9690623512613041
Recall: 0.6346138860749629
F1-score: 0.766963318736168
ROC-AUC: 0.9599207119253799
PR-AUC: 0.9622679612532369
False Negative Rate: 0.365386113925037
False Positive Rate: 0.02677376171352

In [ ]:
# ============================================================
# 17. Train and Evaluate XGBoost
# Default decision threshold = 0.5
# ============================================================

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb_model.fit(X_train_processed, y_train)

xgb_val_score = get_model_scores(xgb_model, X_val_processed)
xgb_test_score = get_model_scores(xgb_model, X_test_processed)

xgb_val_pred = (xgb_val_score >= 0.5).astype(int)
xgb_test_pred = (xgb_test_score >= 0.5).astype(int)

validation_results.append(
    evaluate_model(
        model_name="XGBoost",
        split_name="Validation",
        y_true=y_val,
        y_pred=xgb_val_pred,
        y_score=xgb_val_score,
        threshold=0.5,
        experiment="Baseline"
    )
)

results.append(
    evaluate_model(
        model_name="XGBoost",
        split_name="Test",
        y_true=y_test,
        y_pred=xgb_test_pred,
        y_score=xgb_test_score,
        threshold=0.5,
        experiment="Baseline"
    )
)


Baseline | XGBoost | Validation
Threshold: 0.5
Accuracy: 0.9992458821194682
Precision: 0.9992324093816631
Recall: 0.9991471942691454
F1-score: 0.9991898000085284
ROC-AUC: 0.9999942192320715
PR-AUC: 0.9999935178077874
False Negative Rate: 0.0008528057308545113
False Positive Rate: 0.0006682010542727745

Confusion Matrix:
[[13460     9]
 [   10 11716]]

Classification Report:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     13469
      Attack       1.00      1.00      1.00     11726

    accuracy                           1.00     25195
   macro avg       1.00      1.00      1.00     25195
weighted avg       1.00      1.00      1.00     25195

Baseline | XGBoost | Test
Threshold: 0.5
Accuracy: 0.795333569907736
Precision: 0.9691745633063135
Recall: 0.6614977012389932
F1-score: 0.7863097443497592
ROC-AUC: 0.9664438884719055
PR-AUC: 0.9688420681712901
False Negative Rate: 0.3385022987610068
False Positive Rate: 0.027803521779425393

Con

XGBoost improves over Random Forest on the official test set, especially by catching more attacks and reducing false negatives. However, at the default threshold of 0.5, it still misses a large portion of attacks, so it is not yet the best IDS-oriented configuration.

In [ ]:
# ============================================================
# 18. Train and Evaluate MLP
# Default decision threshold = 0.5
# ============================================================

mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=0.0001,
    batch_size=256,
    learning_rate_init=0.001,
    max_iter=30,
    random_state=RANDOM_STATE,
    verbose=True
)

mlp_model.fit(X_train_processed, y_train)

mlp_val_score = get_model_scores(mlp_model, X_val_processed)
mlp_test_score = get_model_scores(mlp_model, X_test_processed)

mlp_val_pred = (mlp_val_score >= 0.5).astype(int)
mlp_test_pred = (mlp_test_score >= 0.5).astype(int)

validation_results.append(
    evaluate_model(
        model_name="MLP",
        split_name="Validation",
        y_true=y_val,
        y_pred=mlp_val_pred,
        y_score=mlp_val_score,
        threshold=0.5,
        experiment="Baseline"
    )
)

results.append(
    evaluate_model(
        model_name="MLP",
        split_name="Test",
        y_true=y_test,
        y_pred=mlp_test_pred,
        y_score=mlp_test_score,
        threshold=0.5,
        experiment="Baseline"
    )
)


Iteration 1, loss = 0.13421861
Iteration 2, loss = 0.04915419
Iteration 3, loss = 0.03860582
Iteration 4, loss = 0.03353144
Iteration 5, loss = 0.02975864
Iteration 6, loss = 0.02653701
Iteration 7, loss = 0.02418058
Iteration 8, loss = 0.02186694
Iteration 9, loss = 0.02014868
Iteration 10, loss = 0.01848162
Iteration 11, loss = 0.01693509
Iteration 12, loss = 0.01580510
Iteration 13, loss = 0.01526698
Iteration 14, loss = 0.01372782
Iteration 15, loss = 0.01340580
Iteration 16, loss = 0.01274417
Iteration 17, loss = 0.01235678
Iteration 18, loss = 0.01196115
Iteration 19, loss = 0.01134094
Iteration 20, loss = 0.01073343
Iteration 21, loss = 0.01069413
Iteration 22, loss = 0.01059820
Iteration 23, loss = 0.01009057
Iteration 24, loss = 0.01017565
Iteration 25, loss = 0.00962205
Iteration 26, loss = 0.00960721
Iteration 27, loss = 0.00942806
Iteration 28, loss = 0.00927605
Iteration 29, loss = 0.00904388
Iteration 30, loss = 0.00878774
Baseline | MLP | Validation
Threshold: 0.5
Accura

The MLP baseline achieved the best default-threshold test result. On KDDTest+, it reached 82.38% accuracy, 74.85% attack recall, 82.86% F1-score, and 25.15% false negative rate. Compared with Random Forest and XGBoost, it caught more attacks and missed fewer attacks. The tradeoff is that it produced more false positives, with an FPR of 7.67%. Since reducing missed attacks is critical in IDS, MLP is the strongest default-threshold baseline before threshold tuning.

In [ ]:
# ============================================================
# 18.5 Validation-Based Threshold Tuning
# Goal: reduce false negatives without using the official test set for tuning
# Selection metric: F2-score, because IDS cares more about recall than precision
# ============================================================

def find_best_threshold_by_f2(y_val, val_score):
    thresholds = np.arange(0.05, 0.96, 0.01)
    threshold_rows = []

    for threshold in thresholds:
        val_pred = (val_score >= threshold).astype(int)

        precision = precision_score(y_val, val_pred, zero_division=0)
        recall = recall_score(y_val, val_pred, zero_division=0)
        f1 = f1_score(y_val, val_pred, zero_division=0)
        f2 = fbeta_score(y_val, val_pred, beta=2, zero_division=0)

        tn, fp, fn, tp = confusion_matrix(
            y_val,
            val_pred,
            labels=[0, 1]
        ).ravel()

        fnr = fn / (fn + tp) if (fn + tp) else 0.0
        fpr = fp / (fp + tn) if (fp + tn) else 0.0

        threshold_rows.append({
            "Threshold": float(threshold),
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "F2": f2,
            "False Negative Rate": fnr,
            "False Positive Rate": fpr,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,
        })

    threshold_df = pd.DataFrame(threshold_rows)

    best_row = threshold_df.sort_values(
        by=["F2", "Recall", "False Negative Rate", "False Positive Rate"],
        ascending=[False, False, True, True]
    ).iloc[0]

    best_threshold = float(best_row["Threshold"])

    return best_threshold, threshold_df


threshold_tuned_results = []
threshold_search_tables = {}

model_scores = {
    "Random Forest": {
        "val_score": rf_val_score,
        "test_score": rf_test_score,
    },
    "XGBoost": {
        "val_score": xgb_val_score,
        "test_score": xgb_test_score,
    },
    "MLP": {
        "val_score": mlp_val_score,
        "test_score": mlp_test_score,
    },
}

for model_name, scores in model_scores.items():
    best_threshold, threshold_df = find_best_threshold_by_f2(
        y_val=y_val,
        val_score=scores["val_score"]
    )

    threshold_search_tables[model_name] = threshold_df

    print("=" * 70)
    print(f"{model_name} best validation threshold: {best_threshold}")
    print("=" * 70)

    tuned_test_pred = (scores["test_score"] >= best_threshold).astype(int)

    tuned_result = evaluate_model(
        model_name=model_name,
        split_name="Test",
        y_true=y_test,
        y_pred=tuned_test_pred,
        y_score=scores["test_score"],
        threshold=best_threshold,
        experiment="Validation-Tuned Threshold"
    )

    threshold_tuned_results.append(tuned_result)

threshold_tuned_df = pd.DataFrame(threshold_tuned_results)

display(threshold_tuned_df)


Random Forest best validation threshold: 0.35000000000000003
Validation-Tuned Threshold | Random Forest | Test
Threshold: 0.35000000000000003
Accuracy: 0.8098385379701917
Precision: 0.9687362878455463
Recall: 0.6881477440972493
F1-score: 0.8046835846735615
ROC-AUC: 0.9599207119253799
PR-AUC: 0.9622679612532369
False Negative Rate: 0.31185225590275073
False Positive Rate: 0.02934816187828236

Confusion Matrix:
[[9426  285]
 [4002 8831]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.70      0.97      0.81      9711
      Attack       0.97      0.69      0.80     12833

    accuracy                           0.81     22544
   macro avg       0.84      0.83      0.81     22544
weighted avg       0.85      0.81      0.81     22544

XGBoost best validation threshold: 0.34
Validation-Tuned Threshold | XGBoost | Test
Threshold: 0.34
Accuracy: 0.8007452093683464
Precision: 0.968858909499719
Recall: 0.6715499103872827
F1-score: 0.793262150220

,Experiment,Model,Split,Threshold,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,False Negative Rate,False Positive Rate,TN,FP,FN,TP
0,Validation-Tuned Threshold,Random Forest,Test,0.35,0.809839,0.968736,0.688148,0.804684,0.959921,0.962268,0.311852,0.029348,9426,285,4002,8831
1,Validation-Tuned Threshold,XGBoost,Test,0.34,0.800745,0.968859,0.671550,0.793262,0.966444,0.968842,0.328450,0.028524,9434,277,4215,8618
2,Validation-Tuned Threshold,MLP,Test,0.28,0.834147,0.927109,0.769111,0.840751,0.919970,0.934227,0.230889,0.079909,8935,776,2963,9870


I tuned the decision threshold using only the validation set and selected thresholds based on F2-score because IDS prioritizes recall. After applying those thresholds to the official test set, all models improved compared with the default threshold. Random Forest improved its recall from 63.46% to 68.81%, XGBoost from 66.15% to 67.15%, and MLP from 74.85% to 76.91%. The best final model was MLP with threshold 0.28, achieving 83.41% accuracy, 76.91% recall, 84.08% F1-score, and the lowest false negative rate of 23.09%.

In [ ]:
# ============================================================
# 19. Save Baseline Results, Tuned-Threshold Results, and Models
# ============================================================

validation_results_df = pd.DataFrame(validation_results)
results_df = pd.DataFrame(results)
threshold_tuned_df = pd.DataFrame(threshold_tuned_results)

display(validation_results_df)
display(results_df)
display(threshold_tuned_df)

os.makedirs("/content/drive/MyDrive/ML-DP-NID/results", exist_ok=True)
os.makedirs("/content/drive/MyDrive/ML-DP-NID/artifacts", exist_ok=True)

validation_results_df.to_csv(
    "/content/drive/MyDrive/ML-DP-NID/results/validation_results_threshold_0_5.csv",
    index=False
)

results_df.to_csv(
    "/content/drive/MyDrive/ML-DP-NID/results/test_results_threshold_0_5.csv",
    index=False
)

threshold_tuned_df.to_csv(
    "/content/drive/MyDrive/ML-DP-NID/results/test_results_validation_tuned_threshold.csv",
    index=False
)

for model_name, threshold_df in threshold_search_tables.items():
    safe_name = model_name.lower().replace(" ", "_")
    threshold_df.to_csv(
        f"/content/drive/MyDrive/ML-DP-NID/results/{safe_name}_threshold_search_validation.csv",
        index=False
    )

joblib.dump(preprocessor, os.path.join("/content/drive/MyDrive/ML-DP-NID/artifacts", "nsl_kdd_preprocessor.joblib"))
joblib.dump(rf_model, os.path.join("/content/drive/MyDrive/ML-DP-NID/artifacts", "random_forest.joblib"))
joblib.dump(xgb_model, os.path.join("/content/drive/MyDrive/ML-DP-NID/artifacts", "xgboost.joblib"))
joblib.dump(mlp_model, os.path.join("/content/drive/MyDrive/ML-DP-NID/artifacts", "mlp.joblib"))

print("Saved baseline results, tuned-threshold results, threshold search tables, and model artifacts.")


,Experiment,Model,Split,Threshold,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,False Negative Rate,False Positive Rate,TN,FP,FN,TP
0,Baseline,Random Forest,Validation,0.5,0.999047,0.999488,0.998465,0.998976,0.999994,0.999992,0.001535,0.000445,13463,6,18,11708
1,Baseline,XGBoost,Validation,0.5,0.999246,0.999232,0.999147,0.999190,0.999994,0.999994,0.000853,0.000668,13460,9,10,11716
2,Baseline,MLP,Validation,0.5,0.995555,0.994213,0.996248,0.995229,0.999872,0.999862,0.003752,0.005049,13401,68,44,11682


,Experiment,Model,Split,Threshold,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,False Negative Rate,False Positive Rate,TN,FP,FN,TP
0,Baseline,Random Forest,Test,0.5,0.780474,0.969062,0.634614,0.766963,0.959921,0.962268,0.365386,0.026774,9451,260,4689,8144
1,Baseline,XGBoost,Test,0.5,0.795334,0.969175,0.661498,0.786310,0.966444,0.968842,0.338502,0.027804,9441,270,4344,8489
2,Baseline,MLP,Test,0.5,0.823767,0.928019,0.748461,0.828624,0.919970,0.934227,0.251539,0.076717,8966,745,3228,9605


,Experiment,Model,Split,Threshold,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,False Negative Rate,False Positive Rate,TN,FP,FN,TP
0,Validation-Tuned Threshold,Random Forest,Test,0.35,0.809839,0.968736,0.688148,0.804684,0.959921,0.962268,0.311852,0.029348,9426,285,4002,8831
1,Validation-Tuned Threshold,XGBoost,Test,0.34,0.800745,0.968859,0.671550,0.793262,0.966444,0.968842,0.328450,0.028524,9434,277,4215,8618
2,Validation-Tuned Threshold,MLP,Test,0.28,0.834147,0.927109,0.769111,0.840751,0.919970,0.934227,0.230889,0.079909,8935,776,2963,9870


Saved baseline results, tuned-threshold results, threshold search tables, and model artifacts.


In [ ]:
# ============================================================
# 20. Compare Default Threshold vs Validation-Tuned Threshold
# ============================================================

summary_cols = [
    "Experiment",
    "Model",
    "Split",
    "Threshold",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "False Negative Rate",
    "False Positive Rate",
    "ROC_AUC",
    "PR_AUC",
    "TN",
    "FP",
    "FN",
    "TP",
]

print("Default threshold = 0.5 test results")
test_summary_05 = results_df[summary_cols].sort_values(by="F1", ascending=False)
display(test_summary_05)

print("Validation-tuned threshold test results")
tuned_summary = threshold_tuned_df[summary_cols].sort_values(
    by=["False Negative Rate", "F1"],
    ascending=[True, False]
)
display(tuned_summary)

combined_threshold_comparison_df = pd.concat(
    [test_summary_05, tuned_summary],
    ignore_index=True
)

combined_threshold_comparison_df.to_csv(
    "/content/drive/MyDrive/ML-DP-NID/results/default_vs_validation_tuned_threshold_comparison.csv",
    index=False
)

print("Saved results/default_vs_validation_tuned_threshold_comparison.csv")


Default threshold = 0.5 test results


,Experiment,Model,Split,Threshold,Accuracy,Precision,Recall,F1,False Negative Rate,False Positive Rate,ROC_AUC,PR_AUC,TN,FP,FN,TP
2,Baseline,MLP,Test,0.5,0.823767,0.928019,0.748461,0.828624,0.251539,0.076717,0.919970,0.934227,8966,745,3228,9605
1,Baseline,XGBoost,Test,0.5,0.795334,0.969175,0.661498,0.786310,0.338502,0.027804,0.966444,0.968842,9441,270,4344,8489
0,Baseline,Random Forest,Test,0.5,0.780474,0.969062,0.634614,0.766963,0.365386,0.026774,0.959921,0.962268,9451,260,4689,8144


Validation-tuned threshold test results


,Experiment,Model,Split,Threshold,Accuracy,Precision,Recall,F1,False Negative Rate,False Positive Rate,ROC_AUC,PR_AUC,TN,FP,FN,TP
2,Validation-Tuned Threshold,MLP,Test,0.28,0.834147,0.927109,0.769111,0.840751,0.230889,0.079909,0.919970,0.934227,8935,776,2963,9870
0,Validation-Tuned Threshold,Random Forest,Test,0.35,0.809839,0.968736,0.688148,0.804684,0.311852,0.029348,0.959921,0.962268,9426,285,4002,8831
1,Validation-Tuned Threshold,XGBoost,Test,0.34,0.800745,0.968859,0.671550,0.793262,0.328450,0.028524,0.966444,0.968842,9434,277,4215,8618


Saved results/default_vs_validation_tuned_threshold_comparison.csv


After evaluating the baseline and validation-tuned models, I saved all results as CSV files for reproducibility. I saved separate files for validation results, default-threshold test results, tuned-threshold test results, and the full threshold search tables for each model. I also saved the fitted preprocessor and trained models as joblib artifacts. Then I created a final comparison table showing default threshold versus validation-tuned threshold performance, so I can clearly compare how threshold tuning affected recall, F1-score, false negative rate, and false positive rate.

In [ ]:
import json
import platform
from pathlib import Path

ARTIFACT_DIR = Path("/content/drive/MyDrive/ML-DP-NID/artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

repro_manifest = {
    "dataset": {
        "train_file": TRAIN_FILE,
        "test_file": TEST_FILE,
        "data_root": DATA_ROOT,
        "binary_task": "normal_vs_attack",
        "dropped_column": "difficulty",
    },
    "split": {
        "train_rows": int(len(train_part)),
        "val_rows": int(len(val_part)),
        "test_rows": int(len(test_df)),
        "random_state": RANDOM_STATE,
    },
    "environment": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
    "feature_space": {
        "numeric_columns": numeric_cols,
        "categorical_columns": categorical_cols,
        "processed_numeric_dim": int(len(numeric_cols)),
        "processed_total_dim": int(X_train_processed.shape[1]),
    },
}

with open(ARTIFACT_DIR / "repro_manifest.json", "w", encoding="utf-8") as f:
    json.dump(repro_manifest, f, indent=2)

print("Saved repro_manifest.json")

Saved repro_manifest.json


In [ ]:
def export_per_sample_scores(model_name, model, X_split, y_split, split_name, membership, experiment, sigma=None, seed=None, sample_index=None):
    score = get_model_scores(model, X_split)
    if score is None:
        score = np.zeros(len(y_split), dtype=float)
    score = np.asarray(score, dtype=float)
    pred = (score >= 0.5).astype(int)
    y_arr = np.asarray(y_split, dtype=int)
    eps = 1e-8
    safe_score = np.clip(score, eps, 1 - eps)
    loss = -(y_arr * np.log(safe_score) + (1 - y_arr) * np.log(1 - safe_score))
    entropy = -(safe_score * np.log(safe_score) + (1 - safe_score) * np.log(1 - safe_score))
    margin = np.abs(score - 0.5)
    correct = (pred == y_arr).astype(int)
    return pd.DataFrame({
        "experiment": experiment,
        "model": model_name,
        "split": split_name,
        "membership": membership,
        "sigma": sigma,
        "seed": seed,
        "sample_index": sample_index if sample_index is not None else np.arange(len(y_split)),
        "y_true": y_arr,
        "y_pred": pred,
        "score": score,
        "confidence": np.maximum(score, 1 - score),
        "loss": loss,
        "entropy": entropy,
        "margin": margin,
        "correct": correct,
    })

baseline_score_frames = []
baseline_split_specs = [
    ("train", X_train_processed, y_train, "member", train_part.index),
    ("validation", X_val_processed, y_val, "non_member", val_part.index),
    ("test", X_test_processed, y_test, "non_member", test_df.index),
]

for model_name, model in [("Random Forest", rf_model), ("XGBoost", xgb_model), ("MLP", mlp_model)]:
    for split_name, X_split, y_split, membership, sample_index in baseline_split_specs:
        frame = export_per_sample_scores(
            model_name=model_name,
            model=model,
            X_split=X_split,
            y_split=y_split,
            split_name=split_name,
            membership=membership,
            experiment="baseline_clean",
            sigma=0.0,
            seed=RANDOM_STATE,
            sample_index=sample_index,
        )
        baseline_score_frames.append(frame)

baseline_scores_df = pd.concat(baseline_score_frames, ignore_index=True)
baseline_scores_df.to_csv(ARTIFACT_DIR / "baseline_per_sample_scores.csv", index=False)
display(baseline_scores_df.head())
print("Saved baseline_per_sample_scores.csv")


,experiment,model,split,membership,sigma,seed,sample_index,y_true,y_pred,score,confidence,loss,entropy,margin,correct
0,baseline_clean,Random Forest,train,member,0.0,42,23630,0,0,0.0,1.0,1.000000e-08,1.942068e-07,0.5,1
1,baseline_clean,Random Forest,train,member,0.0,42,39958,0,0,0.0,1.0,1.000000e-08,1.942068e-07,0.5,1
2,baseline_clean,Random Forest,train,member,0.0,42,15456,0,0,0.0,1.0,1.000000e-08,1.942068e-07,0.5,1
3,baseline_clean,Random Forest,train,member,0.0,42,169,0,0,0.0,1.0,1.000000e-08,1.942068e-07,0.5,1
4,baseline_clean,Random Forest,train,member,0.0,42,14833,0,0,0.0,1.0,1.000000e-08,1.942068e-07,0.5,1


Saved baseline_per_sample_scores.csv
